# Dispatch Utilities Implementation Notebook

This notebook starts implementation and demonstrates how `utils/aggregated_utils.py` and `utils/dispatch_utils.py` work together.

Workflow covered:
- Build a 48h aggregated forecast table
- Run rule-based dispatch
- Run LP-optimized dispatch
- Validate constraints and compare KPIs
- Execute the full wrapper pipeline

## 1. Set Up Workspace and Dependencies

Install/import required packages, set display options, and verify runtime.

In [ ]:
import sys
import json
import tempfile
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


In [ ]:
# Path bootstrap for notebooks/ and project root execution
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "utils").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"

for p in (PROJECT_ROOT, SRC_DIR):
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

from utils.aggregated_utils import build_aggregated_table, EXPECTED_ROWS_HORIZON
from utils.dispatch_utils import (
    _load_params,
    run_rule_based_dispatch,
    run_lp_dispatch,
    run_balance_checks,
    run_dispatch,
    INTERVAL_MINUTES,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Expected rows: {EXPECTED_ROWS_HORIZON}")
print(f"Interval min : {INTERVAL_MINUTES}")

## 2. Define Configuration and Constants

Centralize paths, thresholds, and defaults so all later cells reuse the same config.

In [ ]:
USER_CONFIG_PATH = PROJECT_ROOT / "user_config.json"
SYSTEM_CONFIG_PATH = PROJECT_ROOT / "system_config.json"
TEMP_AGG_PATH = Path(tempfile.gettempdir()) / "example_dispatch_utils_aggregated.csv"

params = _load_params(USER_CONFIG_PATH, SYSTEM_CONFIG_PATH)

INITIAL_SOC_KWH = float((params["soc_min_kwh"] + params["soc_max_kwh"]) / 2.0)
TOL = 1e-6

print(f"Initial SoC (midpoint): {INITIAL_SOC_KWH:.3f} kWh")
print(f"SoC bounds            : [{params['soc_min_kwh']}, {params['soc_max_kwh']}]")

In [ ]:
with open(USER_CONFIG_PATH, "r", encoding="utf-8") as f:
    user_cfg = json.load(f)
with open(SYSTEM_CONFIG_PATH, "r", encoding="utf-8") as f:
    system_cfg = json.load(f)

print("User config keys:", sorted(user_cfg.keys()))
print("System horizon  :", system_cfg.get("optimization_horizon_hours"), "hours")
print("System interval :", system_cfg.get("interval_minutes"), "minutes")

## 3. Load or Create Sample Input Data

Build an in-memory aggregated 48h table and persist a temporary CSV for wrapper testing.

In [ ]:
try:
    aggregated_df = build_aggregated_table(save_output=False)
    source_mode = "built_live"
except FileNotFoundError as e:
    print("Live build unavailable, falling back to runtime CSV:", e)
    fallback_path = PROJECT_ROOT / "data/runtime/aggregated_table.csv"
    aggregated_df = pd.read_csv(fallback_path)
    source_mode = f"fallback_csv:{fallback_path}"

aggregated_df["utc_timestamp"] = pd.to_datetime(aggregated_df["utc_timestamp"], utc=True)
aggregated_df = aggregated_df.sort_values("utc_timestamp").reset_index(drop=True)
aggregated_df = aggregated_df.iloc[:EXPECTED_ROWS_HORIZON].copy()

aggregated_df.to_csv(TEMP_AGG_PATH, index=False)

print(f"Data source mode: {source_mode}")
print(f"Aggregated shape: {aggregated_df.shape}")
print(f"Temp CSV path  : {TEMP_AGG_PATH}")
print(f"Start          : {aggregated_df['utc_timestamp'].iloc[0]}")
print(f"End            : {aggregated_df['utc_timestamp'].iloc[-1]}")

aggregated_df.head(5)

In [ ]:
# Input data plot (same style as aggregated-table notebook)
plot_df = aggregated_df.copy()
plot_df["utc_timestamp"] = pd.to_datetime(plot_df["utc_timestamp"], utc=True, errors="coerce")
plot_df = plot_df.dropna(subset=["utc_timestamp"]).sort_values("utc_timestamp").reset_index(drop=True)

source_solar = plot_df["source_solar"].dropna().iloc[0] if "source_solar" in plot_df.columns and plot_df["source_solar"].notna().any() else "unknown"
source_load = plot_df["source_load"].dropna().iloc[0] if "source_load" in plot_df.columns and plot_df["source_load"].notna().any() else "unknown"
source_price = plot_df["source_price"].dropna().iloc[0] if "source_price" in plot_df.columns and plot_df["source_price"].notna().any() else "unknown"

fig, ax_left = plt.subplots(figsize=(14, 6))

line_solar = ax_left.plot(
    plot_df["utc_timestamp"],
    plot_df["pv_generation_kwh"],
    marker="o",
    linewidth=2,
    color="#f1c40f",
    label=f"pv_generation_kwh (source: {source_solar})",
)[0]

line_load = ax_left.plot(
    plot_df["utc_timestamp"],
    plot_df["household_load_kwh"],
    marker="o",
    linewidth=2,
    color="#1f77b4",
    label=f"household_load_kwh (source: {source_load})",
)[0]

ax_left.set_xlabel("Timestamp (UTC)")
ax_left.set_ylabel("Energy (kWh)")
ax_left.grid(alpha=0.3)

ax_right = ax_left.twinx()
line_price = ax_right.plot(
    plot_df["utc_timestamp"],
    plot_df["energy_price_buy_cent_kwh"],
    marker="o",
    linewidth=2,
    color="#ff7f0e",
    label=f"energy_price_buy_cent_kwh (source: {source_price})",
)[0]
ax_right.set_ylabel("Price (cent/kWh)")

lines = [line_solar, line_load, line_price]
labels = [line.get_label() for line in lines]
ax_left.legend(lines, labels, title="Metric and source", bbox_to_anchor=(1.02, 1), loc="upper left")

plot_start = plot_df["utc_timestamp"].iloc[0].strftime("%Y-%m-%d")
ax_left.set_title(f"Input data by timestamp with sources ({plot_start}, 48h)")
fig.autofmt_xdate(rotation=45)
plt.tight_layout()
plt.show()

## 4. Implement Core Functions

Run core dispatch methods directly on the prepared aggregated input.

In [ ]:
rule_df = run_rule_based_dispatch(
    forecast_df=aggregated_df,
    params=params,
    actual_soc_kwh=INITIAL_SOC_KWH,
)

lp_df = run_lp_dispatch(
    forecast_df=aggregated_df,
    params=params,
    actual_soc_kwh=INITIAL_SOC_KWH,
)

print("rule_df shape:", rule_df.shape)
print("lp_df shape  :", lp_df.shape)

rule_df[["utc_timestamp", "soc_kwh", "interval_cost_cent", "decision_rule"]].head(5)

## 5. Add Input Validation and Error Handling

Implement checks for row count, continuity, and invalid SoC edge cases.

In [ ]:
assert len(aggregated_df) == EXPECTED_ROWS_HORIZON, (
    f"Expected {EXPECTED_ROWS_HORIZON} rows, got {len(aggregated_df)}"
)

delta = aggregated_df["utc_timestamp"].diff().dropna()
assert (delta == pd.Timedelta(minutes=INTERVAL_MINUTES)).all(), "Timestamp spacing is not continuous"

bad_soc = params["soc_max_kwh"] + 1.0
try:
    run_dispatch(
        actual_soc_kwh=bad_soc,
        aggregated_csv=TEMP_AGG_PATH,
        user_config_path=USER_CONFIG_PATH,
        system_config_path=SYSTEM_CONFIG_PATH,
        save_output=False,
    )
    raise AssertionError("Expected ValueError for invalid SoC, but none was raised")
except ValueError as e:
    print("Caught expected ValueError:", e)

## 6. Compose an End-to-End Execution Pipeline

Create a single function that executes aggregate + rule + LP + checks + wrapper.

In [ ]:
def run_full_pipeline(initial_soc_kwh: float) -> dict:
    try:
        agg = build_aggregated_table(save_output=False)
        source_mode = "built_live"
    except FileNotFoundError:
        fallback_path = PROJECT_ROOT / "data/runtime/aggregated_table.csv"
        agg = pd.read_csv(fallback_path)
        source_mode = f"fallback_csv:{fallback_path}"

    agg["utc_timestamp"] = pd.to_datetime(agg["utc_timestamp"], utc=True)
    agg = agg.sort_values("utc_timestamp").reset_index(drop=True)
    agg = agg.iloc[:EXPECTED_ROWS_HORIZON].copy()
    agg.to_csv(TEMP_AGG_PATH, index=False)

    rule = run_rule_based_dispatch(agg, params, initial_soc_kwh)
    lp = run_lp_dispatch(agg, params, initial_soc_kwh)

    checks_rule = run_balance_checks(rule, params)
    checks_lp = run_balance_checks(lp, params)

    combined = run_dispatch(
        actual_soc_kwh=initial_soc_kwh,
        aggregated_csv=TEMP_AGG_PATH,
        user_config_path=USER_CONFIG_PATH,
        system_config_path=SYSTEM_CONFIG_PATH,
        save_output=False,
    )

    return {
        "source_mode": source_mode,
        "aggregated": agg,
        "rule": rule,
        "lp": lp,
        "checks_rule": checks_rule,
        "checks_lp": checks_lp,
        "combined": combined,
    }

## 7. Write Quick Unit Tests in Notebook Cells

Add fast checks for normal and edge behavior.

In [ ]:
assert len(rule_df) == EXPECTED_ROWS_HORIZON
assert len(lp_df) == EXPECTED_ROWS_HORIZON

rule_checks = run_balance_checks(rule_df, params)
lp_checks = run_balance_checks(lp_df, params)

assert bool(rule_checks["soc_within_bounds"]) is True
assert bool(lp_checks["soc_within_bounds"]) is True

assert "interval_cost_cent" in rule_df.columns
assert "interval_cost_cent" in lp_df.columns

print("Quick tests passed.")

## 8. Run the Implementation and Inspect Outputs

Execute the composed pipeline and inspect key metrics and outputs.

In [ ]:
pipeline_out = run_full_pipeline(INITIAL_SOC_KWH)

kpi_df = pd.DataFrame(
    {
        "metric": ["cost_cent", "grid_import_kwh", "grid_export_kwh", "final_soc_kwh"],
        "rule_based": [
            float(pipeline_out["rule"]["interval_cost_cent"].sum()),
            float(pipeline_out["rule"]["total_import_kwh"].sum()),
            float(pipeline_out["rule"]["total_export_kwh"].sum()),
            float(pipeline_out["rule"]["soc_kwh"].iloc[-1]),
        ],
        "lp_optimized": [
            float(pipeline_out["lp"]["interval_cost_cent"].sum()),
            float(pipeline_out["lp"]["total_import_kwh"].sum()),
            float(pipeline_out["lp"]["total_export_kwh"].sum()),
            float(pipeline_out["lp"]["soc_kwh"].iloc[-1]),
        ],
    }
)
kpi_df["delta_lp_minus_rule"] = kpi_df["lp_optimized"] - kpi_df["rule_based"]

print("Combined output shape:", pipeline_out["combined"].shape)
kpi_df

### Additional Plot: Step-Style Dispatch Overview

The plot below shows the dispatch decisions over time, highlighting when the system is charging, discharging, or idle. This visual representation helps to quickly understand the behavior of the dispatch algorithm across the 48-hour forecast horizon.


In [ ]:
def plot_step_dispatch_overview(
    df: pd.DataFrame,
    params: dict | None = None,
    title: str = "Dispatch / Forecast Overview",
    timestamp_col: str = "utc_timestamp",
) -> None:
    import matplotlib.ticker as mticker

    plot_df = df.copy()

    if timestamp_col in plot_df.columns:
        plot_df[timestamp_col] = pd.to_datetime(plot_df[timestamp_col], utc=True, errors="coerce")
        if plot_df[timestamp_col].isna().any():
            raise ValueError(f"Column '{timestamp_col}' contains invalid timestamps.")
        x_axis = (plot_df[timestamp_col] - plot_df[timestamp_col].iloc[0]).dt.total_seconds() / 3600.0
        x_label = "Hour of day (UTC)"
    else:
        x_axis = np.arange(len(plot_df))
        x_label = "Row index"

    fig, (ax_flow, ax_soc, ax_cost) = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    fig.suptitle(title, fontsize=13, fontweight="bold")

    flow_specs = [
        ("household_load_kwh", "Household load", "#1f77b4", "plot"),
        ("pv_generation_kwh", "PV generation", "#f4a216", "plot"),
        ("grid_to_load_kwh", "Grid to load", "#d62728", "step"),
        ("battery_to_load_kwh", "Battery to load", "#2ca02c", "step"),
        ("pv_to_battery_kwh", "PV to battery", "#17becf", "step"),
        ("grid_to_battery_kwh", "Grid to battery", "#9467bd", "step"),
        ("export_to_grid_kwh", "Export to grid", "#8c564b", "step"),
    ]

    has_flows = False
    for col, label, color, mode in flow_specs:
        if col not in plot_df.columns:
            continue
        if mode == "step":
            ax_flow.step(x_axis, plot_df[col], where="post", linewidth=1.3, color=color, label=label)
        else:
            ax_flow.plot(x_axis, plot_df[col], linewidth=1.8, color=color, label=label)
        has_flows = True

    if has_flows:
        ax_flow.legend(loc="upper right", fontsize=8, ncol=2)
    else:
        ax_flow.text(0.5, 0.5, "No flow columns found.", ha="center", va="center", transform=ax_flow.transAxes)
    ax_flow.set_ylabel("kWh per 15 min")

    soc_col = "soc_kwh" if "soc_kwh" in plot_df.columns else ("soc_percent" if "soc_percent" in plot_df.columns else None)
    if soc_col:
        soc_unit = "kWh" if soc_col == "soc_kwh" else "%"
        ax_soc.plot(x_axis, plot_df[soc_col], color="#8c564b", linewidth=2.0, label=f"SoC ({soc_unit})")
        if soc_col == "soc_kwh" and params and "soc_min_kwh" in params and "soc_max_kwh" in params:
            ax_soc.axhline(params["soc_min_kwh"], color="#888", ls="--", lw=1.0, label="SoC min/max")
            ax_soc.axhline(params["soc_max_kwh"], color="#888", ls="--", lw=1.0)
        ax_soc.legend(loc="upper right", fontsize=8)
        ax_soc.set_ylabel(f"SoC ({soc_unit})")
    else:
        ax_soc.text(0.5, 0.5, "No SoC columns found.", ha="center", va="center", transform=ax_soc.transAxes)
        ax_soc.set_ylabel("SoC")

    has_cost = False
    if "cumulative_cost_cent" in plot_df.columns:
        ax_cost.plot(x_axis, plot_df["cumulative_cost_cent"], color="#9467bd", linewidth=2.0, label="Cumulative cost (cent)")
        has_cost = True
    elif "interval_cost_cent" in plot_df.columns:
        ax_cost.plot(x_axis, plot_df["interval_cost_cent"].cumsum(), color="#9467bd", linewidth=2.0, label="Cumulative cost (cent)")
        has_cost = True

    price_col = next((c for c in ["predicted_energy_price_buy_cent_kwh", "energy_price_buy_cent_kwh"] if c in plot_df.columns), None)
    sell_price = float(params["default_sell_price_cent_kwh"]) if params and "default_sell_price_cent_kwh" in params else None

    if price_col or sell_price is not None:
        ax_price = ax_cost.twinx()
        if price_col:
            ax_price.plot(x_axis, plot_df[price_col], color="#ff7f0e", lw=1.8, ls="--", label="Buy price (cent/kWh)")
        if sell_price is not None:
            ax_price.axhline(sell_price, color="#2ca02c", lw=1.5, ls=":", label="Sell price (cent/kWh)")
        ax_price.legend(loc="upper right", fontsize=8)
        ax_price.set_ylabel("Price (cent/kWh)")
        ax_price.grid(False)

    ax_cost.set_xlabel(x_label)
    ax_cost.set_ylabel("cent")
    if has_cost:
        ax_cost.legend(loc="upper left", fontsize=8)

    for ax in (ax_flow, ax_soc, ax_cost):
        ax.grid(True, linestyle="--", alpha=0.35, which="both")
        ax.xaxis.grid(True, linestyle="--", alpha=0.85, which="major")
        ax.set_xlim(0, 48)

    ax_cost.xaxis.set_major_locator(mticker.MultipleLocator(3))
    ax_cost.xaxis.set_minor_locator(mticker.MultipleLocator(1))

    fig.tight_layout()
    plt.show()

In [ ]:
plot_step_dispatch_overview(
    pipeline_out["lp"],
    params=params,
    title="LP Dispatch Overview (step-style)",
)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(pipeline_out["rule"]["utc_timestamp"], pipeline_out["rule"]["soc_kwh"], label="rule_based", linewidth=1.8)
axes[0].plot(pipeline_out["lp"]["utc_timestamp"], pipeline_out["lp"]["soc_kwh"], label="lp_optimized", linewidth=1.8)
axes[0].axhline(params["soc_min_kwh"], linestyle="--", alpha=0.6)
axes[0].axhline(params["soc_max_kwh"], linestyle="--", alpha=0.6)
axes[0].set_ylabel("SoC (kWh)")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(pipeline_out["rule"]["utc_timestamp"], pipeline_out["rule"]["cumulative_cost_cent"], label="rule_based", linewidth=1.8)
axes[1].plot(pipeline_out["lp"]["utc_timestamp"], pipeline_out["lp"]["cumulative_cost_cent"], label="lp_optimized", linewidth=1.8)
axes[1].set_ylabel("Cumulative cost (cent)")
axes[1].set_xlabel("UTC timestamp")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 9. Add Basic Logging and Performance Checks

Measure runtime and print compact execution logs for reproducible profiling.

In [ ]:
print("[LOG] Starting timed pipeline run...")
t0 = perf_counter()
perf_out = run_full_pipeline(INITIAL_SOC_KWH)
elapsed = perf_counter() - t0

print(f"[LOG] Timed pipeline finished in {elapsed:.3f} s")
print(f"[LOG] rows aggregated/rule/lp/combined: {len(perf_out['aggregated'])}/{len(perf_out['rule'])}/{len(perf_out['lp'])}/{len(perf_out['combined'])}")
print(f"[LOG] LP total cost (cent): {perf_out['lp']['interval_cost_cent'].sum():.2f}")
print(f"[LOG] Rule total cost (cent): {perf_out['rule']['interval_cost_cent'].sum():.2f}")